# Evaluate Semantic View: LIQUIDITY_SV

This notebook validates the `LIQUIDITY_SV` semantic view by running checks against its underlying tables, columns, relationships, and verified queries.

**Connection**: Uses the `mcastro` connection from `~/.snowflake/connections.toml`.

In [ ]:
from snowflake.snowpark import Session
from pathlib import Path
from cryptography.hazmat.primitives import serialization
import json

private_key_path = Path.home() / ".ssh" / "mlops_hol_rsa_private_key.pem"
private_key = serialization.load_pem_private_key(private_key_path.read_bytes(), password=None)
private_key_der = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

session = Session.builder.configs({
    "account": "SFSEEUROPE-MCASTRO_AWS1_USWEST2",
    "user": "mcastro",
    "authenticator": "SNOWFLAKE_JWT",
    "private_key": private_key_der,
    "database": "LIQUIDITY_RISK_DB",
    "schema": "PUBLIC",
    "warehouse": "LIQUIDITY_RISK_WH",
    "role": "LIQUIDITY_RISK_ROLE",
}).create()

DB = "LIQUIDITY_RISK_DB"
SCHEMA = "PUBLIC"
SV_NAME = "LIQUIDITY_SV"
SV_FQN = f"{DB}.{SCHEMA}.{SV_NAME}"

SV_TABLES = {
    "ASSET_CLASSIFICATIONS":      f"{DB}.RAW.ASSET_CLASSIFICATIONS",
    "BUSINESS_UNIT_REFERENCE":    f"{DB}.RAW.BUSINESS_UNIT_REFERENCE",
    "CASH_INFLOWS":               f"{DB}.RAW.CASH_INFLOWS",
    "CASH_OUTFLOWS":              f"{DB}.RAW.CASH_OUTFLOWS",
    "COUNTERPARTY_DATA":          f"{DB}.RAW.COUNTERPARTY_DATA",
    "HQLA_SUMMARY":               f"{DB}.RAW.HQLA_SUMMARY",
    "POSITIONS":                  f"{DB}.RAW.POSITIONS",
    "LCR":                        f"{DB}.PRESENTATION.LCR",
    "WHAT_IF_DEFINITIONS_LOOKUP": f"{DB}.RAW.WHAT_IF_DEFINITIONS_LOOKUP",
    "WHAT_IF_LCR":                f"{DB}.PRESENTATION.WHAT_IF_LCR",
}

SV_RELATIONSHIPS = [
    {"name": "INFLOW_COUNTERPARTY",             "left": "CASH_INFLOWS",  "right": "COUNTERPARTY_DATA",      "left_col": "COUNTERPARTY_ID", "right_col": "COUNTERPARTY_ID"},
    {"name": "OUTFLOW_BUSINESS_UNIT",            "left": "CASH_OUTFLOWS", "right": "BUSINESS_UNIT_REFERENCE", "left_col": "BUSINESS_UNIT_ID", "right_col": "BUSINESS_UNIT_ID"},
    {"name": "OUTFLOW_COUNTERPARTY",             "left": "CASH_OUTFLOWS", "right": "COUNTERPARTY_DATA",      "left_col": "COUNTERPARTY_ID", "right_col": "COUNTERPARTY_ID"},
    {"name": "POSITIONS_TO_ASSET_SECURITY_TYPE", "left": "POSITIONS",     "right": "ASSET_CLASSIFICATIONS",  "left_col": "SECURITY_TYPE",   "right_col": "SECURITY_TYPE"},
    {"name": "POSITION_BUSINESS_UNIT",           "left": "POSITIONS",     "right": "BUSINESS_UNIT_REFERENCE", "left_col": "BUSINESS_UNIT_ID", "right_col": "BUSINESS_UNIT_ID"},
]

results = {}
print(f"Connected. Evaluating: {SV_FQN}")
print(f"Tables: {len(SV_TABLES)} | Relationships: {len(SV_RELATIONSHIPS)}")

In [ ]:
# ============================================
# CHECK 1: Semantic View Exists
# ============================================
print("=" * 60)
print("CHECK 1: Semantic View Exists")
print("=" * 60)

sv_rows = session.sql(f"SHOW SEMANTIC VIEWS LIKE '{SV_NAME}' IN SCHEMA {DB}.{SCHEMA}").collect()

if len(sv_rows) > 0:
    results["Semantic View Exists"] = "PASS"
    print(f"PASS - Found: {SV_NAME}")
    for row in sv_rows:
        print(f"  Created:  {row['created_on']}")
        print(f"  Database: {row['database_name']}")
        print(f"  Schema:   {row['schema_name']}")
else:
    results["Semantic View Exists"] = "FAIL"
    print(f"FAIL - '{SV_NAME}' not found")

print()
desc_rows = session.sql(f"DESC SEMANTIC VIEW {SV_FQN}").collect()
print(f"DESC returned {len(desc_rows)} rows")

In [ ]:
# ============================================
# CHECK 2: Table Coverage
# ============================================
print("=" * 60)
print("CHECK 2: Table Coverage - All referenced tables exist & have data")
print("=" * 60)

table_counts = {}
all_tables_ok = True

for sv_name, fqn in SV_TABLES.items():
    try:
        count = session.sql(f"SELECT COUNT(*) AS CNT FROM {fqn}").collect()[0]["CNT"]
        table_counts[sv_name] = count
        status = "PASS" if count > 0 else "WARN - empty"
        if count == 0:
            all_tables_ok = False
        print(f"  {status:20s} | {sv_name:30s} | {count:>10,} rows")
    except Exception as e:
        table_counts[sv_name] = -1
        all_tables_ok = False
        print(f"  {'FAIL':20s} | {sv_name:30s} | {e}")

results["All Tables Exist & Have Data"] = "PASS" if all_tables_ok else "FAIL"
print(f"\nOverall: {'PASS' if all_tables_ok else 'FAIL'}")

In [ ]:
# ============================================
# CHECK 3: Column Validation
# Dynamically reads columns from DESC SEMANTIC VIEW
# ============================================
print("=" * 60)
print("CHECK 3: Column Validation - SV columns exist in actual tables")
print("=" * 60)

desc_rows = session.sql(f"DESC SEMANTIC VIEW {SV_FQN}").collect()

sv_columns_map = {}
for row in desc_rows:
    kind = row["object_kind"]
    if kind in ("DIMENSION", "FACT"):
        table_name = row["parent_entity"]
        col_name = row["object_name"]
        if table_name not in sv_columns_map:
            sv_columns_map[table_name] = set()
        sv_columns_map[table_name].add(col_name)

all_columns_ok = True

for sv_name, expected_cols in sv_columns_map.items():
    if sv_name not in SV_TABLES:
        print(f"  SKIP | {sv_name}: not in SV_TABLES lookup")
        continue
    fqn = SV_TABLES[sv_name]
    try:
        actual_cols_rows = session.sql(f"DESCRIBE TABLE {fqn}").collect()
        actual_cols = {row['name'] for row in actual_cols_rows}
        missing = expected_cols - actual_cols
        extra = actual_cols - expected_cols

        if missing:
            all_columns_ok = False
            print(f"  FAIL | {sv_name}: Missing in table: {missing}")
        else:
            print(f"  PASS | {sv_name}: All {len(expected_cols)} SV columns found")
        if extra:
            print(f"         Info: {len(extra)} extra col(s) in table not in SV: {extra}")
    except Exception as e:
        all_columns_ok = False
        print(f"  FAIL | {sv_name}: {e}")

results["All SV Columns Exist in Tables"] = "PASS" if all_columns_ok else "FAIL"
print(f"\nOverall: {'PASS' if all_columns_ok else 'FAIL'}")

In [ ]:
# ============================================
# CHECK 4: Relationship Validation
# ============================================
print("=" * 60)
print("CHECK 4: Relationship Validation - FK joins produce no orphans")
print("=" * 60)

all_rels_ok = True

for rel in SV_RELATIONSHIPS:
    left_fqn = SV_TABLES[rel["left"]]
    right_fqn = SV_TABLES[rel["right"]]
    left_col = rel["left_col"]
    right_col = rel["right_col"]

    orphan_query = f"""
        SELECT COUNT(*) AS orphan_count
        FROM {left_fqn} l
        LEFT JOIN {right_fqn} r ON l.{left_col} = r.{right_col}
        WHERE r.{right_col} IS NULL
    """
    try:
        orphan_count = session.sql(orphan_query).collect()[0]["ORPHAN_COUNT"]
        left_total = session.sql(f"SELECT COUNT(*) AS CNT FROM {left_fqn}").collect()[0]["CNT"]

        if orphan_count == 0:
            print(f"  PASS | {rel['name']}: 0 orphans / {left_total:,} rows")
        else:
            all_rels_ok = False
            pct = orphan_count / left_total * 100 if left_total > 0 else 0
            print(f"  WARN | {rel['name']}: {orphan_count:,} orphans / {left_total:,} rows ({pct:.1f}%)")
    except Exception as e:
        all_rels_ok = False
        print(f"  FAIL | {rel['name']}: {e}")

results["All Relationships Valid"] = "PASS" if all_rels_ok else "FAIL"
print(f"\nOverall: {'PASS' if all_rels_ok else 'FAIL'}")

In [ ]:
# ============================================
# CHECK 5: Verified Query Testing
# ============================================
print("=" * 60)
print("CHECK 5: Verified Queries from the SV definition")
print("=" * 60)

VERIFIED_QUERIES = [
    {
        "name": "LCR Trend Over Time",
        "sql": f"""
            SELECT lcr.day_number, lcr.total_net_cash_outflows, lcr.hqla, lcr.lcr
            FROM {DB}.PRESENTATION.LCR AS lcr
            WHERE lcr.created_timestamp IN (SELECT MAX(created_timestamp) FROM {DB}.PRESENTATION.LCR)
            ORDER BY lcr.day_number ASC NULLS FIRST
        """
    },
    {
        "name": "LCR Change Analysis (LAG)",
        "sql": f"""
            SELECT day_number, hqla, total_net_cash_outflows, lcr,
                hqla - LAG(hqla) OVER (ORDER BY day_number) AS hqla_change,
                total_net_cash_outflows - LAG(total_net_cash_outflows) OVER (ORDER BY day_number) AS outflows_change,
                lcr - LAG(lcr) OVER (ORDER BY day_number) AS lcr_change
            FROM {DB}.PRESENTATION.LCR
            WHERE day_number <= 150
                AND CREATED_TIMESTAMP = (SELECT MAX(CREATED_TIMESTAMP) FROM {DB}.PRESENTATION.LCR)
            ORDER BY day_number ASC
        """
    }
]

all_queries_ok = True

for vq in VERIFIED_QUERIES:
    print(f"\n--- {vq['name']} ---")
    try:
        rows = session.sql(vq["sql"]).collect()
        if len(rows) > 0:
            print(f"  PASS - {len(rows)} rows")
            print(f"  First: {rows[0].as_dict()}")
            if len(rows) > 1:
                print(f"  Last:  {rows[-1].as_dict()}")
        else:
            all_queries_ok = False
            print(f"  WARN - 0 rows (LCR table may need to be populated first)")
    except Exception as e:
        all_queries_ok = False
        print(f"  FAIL - {e}")

results["Verified Queries Execute"] = "PASS" if all_queries_ok else "FAIL"
print(f"\nOverall: {'PASS' if all_queries_ok else 'FAIL'}")

In [ ]:
# ============================================
# CHECK 6: Cortex Analyst Integration Test
# Uses REST API via snowflake.connector
# ============================================
print("=" * 60)
print("CHECK 6: Cortex Analyst - Natural language queries against SV")
print("=" * 60)

import requests

test_questions = [
    "What is the latest LCR value for day 1?",
    "How many counterparties are there by type?",
    "What is the total HQLA for the first 30 days?",
]

analyst_ok = True
connection = session.connection
account = connection.account
token = connection.rest.token

# Build the REST host from the account identifier
host = connection.host
url = f"https://{host}/api/v2/cortex/analyst/message"

headers = {
    "Authorization": f"Snowflake Token=\"{token}\"",
    "Content-Type": "application/json",
}

for question in test_questions:
    print(f"\n--- Question: {question} ---")
    try:
        payload = {
            "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
            "semantic_view": SV_FQN,
        }
        resp = requests.post(url, headers=headers, json=payload, timeout=30)
        resp.raise_for_status()
        resp_json = resp.json()
        message = resp_json.get("message", {})
        content_items = message.get("content", [])

        generated_sql = None
        text_response = None
        for item in content_items:
            if item.get("type") == "sql":
                generated_sql = item.get("statement", "")
            elif item.get("type") == "text":
                text_response = item.get("text", "")

        if generated_sql:
            preview = generated_sql[:200] + "..." if len(generated_sql) > 200 else generated_sql
            print(f"  SQL: {preview}")
            try:
                result_rows = session.sql(generated_sql).collect()
                print(f"  PASS - {len(result_rows)} rows returned")
                if result_rows:
                    print(f"  Sample: {result_rows[0].as_dict()}")
            except Exception as sql_err:
                analyst_ok = False
                print(f"  FAIL - SQL execution error: {sql_err}")
        elif text_response:
            print(f"  Text: {text_response[:300]}")
        else:
            analyst_ok = False
            print(f"  WARN - Unexpected response")

    except Exception as e:
        analyst_ok = False
        print(f"  FAIL - {e}")

results["Cortex Analyst Integration"] = "PASS" if analyst_ok else "FAIL"
print(f"\nOverall: {'PASS' if analyst_ok else 'FAIL'}")

In [ ]:
# ============================================
# Summary Report
# ============================================
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"{'Check':<45} {'Result':>10}")
print("-" * 57)

pass_count = 0
fail_count = 0

for check, result in results.items():
    icon = "[PASS]" if result == "PASS" else "[FAIL]"
    print(f"  {check:<43} {icon:>10}")
    if result == "PASS":
        pass_count += 1
    else:
        fail_count += 1

print("-" * 57)
total = pass_count + fail_count
print(f"  Total: {pass_count}/{total} checks passed")
print()

if fail_count == 0:
    print("  All checks passed. The semantic view is correctly configured.")
else:
    print(f"  {fail_count} check(s) failed. Review the details above.")

session.close()